### Structural Novelty

- STEP 3A — Structural Novelty (Temporal KG-Based)
- For each paper (in chronological order):
- Structural Novelty = (# triples not seen before) / (total triples in paper)
- Historical triples are accumulated strictly by year.

In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [4]:
# Load knowledge edges
edges = pd.read_csv("../../outputs/knowledge_edges.csv")
edges = edges.dropna(subset=['year'])

# Ensure correct types
edges["year"] = edges["year"].astype(int)

# Sort strictly by year
edges = edges.sort_values("year")

print("Total triples:", len(edges))
print("Year range:", edges["year"].min(), "-", edges["year"].max())

Total triples: 234348
Year range: 2010 - 2020


In [5]:
historical_triples = set()
structural_scores = []

# Get papers grouped by year
years_sorted = sorted(edges["year"].unique())

for year in tqdm(years_sorted):

    year_edges = edges[edges["year"] == year]

    # Get unique papers in this year
    papers_in_year = year_edges["source"].unique()

    # First score all papers in this year (without updating history)
    year_paper_triples = {}

    for paper_id in papers_in_year:
        paper_edges = year_edges[year_edges["source"] == paper_id]
        triples = set(zip(paper_edges["predicate"],
                          paper_edges["target"]))

        if len(triples) == 0:
            score = 0.0
        else:
            new_triples = sum(1 for t in triples if t not in historical_triples)
            score = new_triples / len(triples)          ## structual novelty scoring

        structural_scores.append((paper_id, score, len(triples)))
        year_paper_triples[paper_id] = triples

    # After scoring all papers in this year, update history
    for triples in year_paper_triples.values():
        historical_triples.update(triples)

struct_df = pd.DataFrame(structural_scores,
                         columns=["paper_id",
                                  "structural_novelty",
                                  "triple_count"])

100%|██████████| 11/11 [00:04<00:00,  2.39it/s]


In [6]:
struct_df.to_csv("../../outputs/structural_novelty_scores.csv", index=False)

In [7]:
print(struct_df["structural_novelty"].describe())

count    2116.000000
mean        0.769444
std         0.098681
min         0.000000
25%         0.709302
50%         0.769813
75%         0.825770
max         1.000000
Name: structural_novelty, dtype: float64
